# Phase 8 — Nonlinear Model Selection

## TL;DR

- Random forest is the strongest initial validation candidate with MAE of about 10.2k hg/ha, RMSE of 19.6k hg/ha, and R² of 0.947.
- It reduces MAE by about 66.7% relative to ordinary linear regression and produces no negative validation predictions.
- Poisson gradient boosting ranks second with MAE of about 14.0k hg/ha and positive predictions.
- Log-target ridge regression improves on ordinary linear regression and prevents negative predictions, but remains weaker than the tree models.
- Random forest errors are not equally distributed: cassava, potatoes, and sweet potatoes have the largest crop-level MAE.
- Test data remains untouched; random forest is a candidate for tuning, not yet the final model.


## Context & Methods

The experiment compares five approaches on 2008–2010 validation data after fitting only on 1990–2007 training data:

1. crop-median benchmark;
2. ordinary linear regression;
3. log-target ridge regression;
4. random forest; and
5. Poisson histogram gradient boosting.

The candidate configurations are fixed before this comparison. Hyperparameter tuning is a later phase.

### Key assumptions

- Lower validation MAE is the primary selection criterion.
- RMSE, R², physical plausibility, crop-level errors, and time stability are supporting criteria.
- Validation results choose the next candidate; test results do not influence development.
- Predictive performance does not establish causal agricultural effects.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from crop_yield.evaluation import (
    prediction_diagnostics,
    regression_metrics,
)
from crop_yield.models import (
    CropMedianRegressor,
    build_linear_regression_pipeline,
    build_log_target_ridge_pipeline,
    build_poisson_gradient_boosting_pipeline,
    build_random_forest_pipeline,
)
from crop_yield.preprocessing import split_features_target
from crop_yield.splitting import temporal_train_validation_test_split

plt.style.use("seaborn-v0_8-whitegrid")

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_PATH = REPO_ROOT / "data/processed/crop_yield_modeling.csv"


## Data

### 1. Load training and validation partitions


In [ ]:
crop_yield = pd.read_csv(DATA_PATH)
temporal_split = temporal_train_validation_test_split(crop_yield)

X_train, y_train = split_features_target(temporal_split.train)
X_validation, y_validation = split_features_target(
    temporal_split.validation
)

print(f"Training rows: {len(X_train):,}")
print(f"Validation rows: {len(X_validation):,}")
print(f"Untouched test rows: {len(temporal_split.test):,}")


## Results

### 2. Fit the candidate models


In [ ]:
candidate_models = {
    "Crop median": CropMedianRegressor(),
    "Linear regression": build_linear_regression_pipeline(),
    "Log-target ridge": build_log_target_ridge_pipeline(),
    "Random forest": build_random_forest_pipeline(),
    "Poisson gradient boosting": (
        build_poisson_gradient_boosting_pipeline()
    ),
}

validation_predictions = {}
model_results = []

for model_name, model in candidate_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_validation)
    validation_predictions[model_name] = predictions
    model_results.append(
        {
            "model": model_name,
            **regression_metrics(y_validation, predictions),
            **prediction_diagnostics(predictions),
        }
    )

results = (
    pd.DataFrame(model_results)
    .set_index("model")
    .sort_values("mae")
)
results


### 3. Compare validation performance


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

results.sort_values("mae")["mae"].plot.barh(
    ax=axes[0],
    color="#3B6EA8",
)
axes[0].set_title("Validation MAE by Model")
axes[0].set_xlabel("Mean absolute error (hg/ha)")
axes[0].set_ylabel("")

results.sort_values("r2")["r2"].plot.barh(
    ax=axes[1],
    color="#8FA8C7",
)
axes[1].set_title("Validation R² by Model")
axes[1].set_xlabel("R²")
axes[1].set_ylabel("")

fig.suptitle("Initial Nonlinear Model Comparison", fontsize=16)
fig.tight_layout()
plt.show()


### 4. Measure improvement over linear regression


In [ ]:
linear_mae = results.loc["Linear regression", "mae"]
forest_mae = results.loc["Random forest", "mae"]
forest_improvement = (linear_mae - forest_mae) / linear_mae

print(
    "Random forest reduces MAE relative to ordinary linear regression "
    f"by {forest_improvement:.1%}."
)


### 5. Analyze random-forest errors by crop

An overall average can hide weak performance for specific agricultural systems. Crop-level MAE identifies where the candidate still needs improvement.


In [ ]:
forest_error_data = temporal_split.validation.loc[
    :, ["area", "item", "year", "yield_hg_per_ha"]
].copy()
forest_error_data["prediction"] = validation_predictions[
    "Random forest"
]
forest_error_data["absolute_error"] = (
    forest_error_data["yield_hg_per_ha"]
    - forest_error_data["prediction"]
).abs()

crop_errors = (
    forest_error_data.groupby("item")["absolute_error"]
    .agg(observations="size", mae="mean", median_absolute_error="median")
    .sort_values("mae", ascending=False)
)
crop_errors


In [ ]:
fig, axis = plt.subplots(figsize=(10, 6))
crop_errors.sort_values("mae")["mae"].plot.barh(
    ax=axis,
    color="#3B6EA8",
)
axis.set_title("Random-Forest Validation MAE by Crop")
axis.set_xlabel("Mean absolute error (hg/ha)")
axis.set_ylabel("")
plt.tight_layout()
plt.show()


### 6. Check error stability across validation years


In [ ]:
year_errors = (
    forest_error_data.groupby("year")["absolute_error"]
    .agg(observations="size", mae="mean", median_absolute_error="median")
)

error_coverage = pd.Series({
    "share_within_10k_hg_per_ha": (
        forest_error_data["absolute_error"].le(10_000).mean()
    ),
    "share_within_20k_hg_per_ha": (
        forest_error_data["absolute_error"].le(20_000).mean()
    ),
    "median_absolute_error": forest_error_data[
        "absolute_error"
    ].median(),
})

print(year_errors)
error_coverage


### 7. Preserve the test set


In [ ]:
test_evaluated = False
print(
    "No candidate has been evaluated on the test set. "
    f"Reserved observations: {len(temporal_split.test):,}."
)


## Checks


In [ ]:
assert results.index[0] == "Random forest"
assert np.isclose(results.loc["Random forest", "mae"], 10_194.89, atol=2.0)
assert np.isclose(results.loc["Random forest", "r2"], 0.9471, atol=0.001)
assert np.isclose(
    results.loc["Poisson gradient boosting", "mae"],
    13_985.98,
    atol=2.0,
)
assert np.isclose(results.loc["Log-target ridge", "mae"], 25_646.76, atol=2.0)
assert results.loc["Random forest", "negative_prediction_count"] == 0
assert results.loc["Log-target ridge", "negative_prediction_count"] == 0
assert results.loc["Poisson gradient boosting", "negative_prediction_count"] == 0
assert crop_errors.index[0] == "Cassava"
assert test_evaluated is False
print("All Phase 8 model-selection checks passed.")


## Takeaways

1. Random forest is the strongest initial validation candidate and produces physically plausible non-negative predictions.
2. Its validation MAE is about 10.2k hg/ha, and 72.6% of validation predictions are within 10k hg/ha of observed yield.
3. Cassava, potatoes, and sweet potatoes have materially larger errors than most grain crops, so overall performance must not be generalized equally to every crop.
4. Validation MAE rises from 2008 through 2010, which may indicate increasing temporal difficulty and should be examined during tuning.
5. The next phase will tune the random forest using training-era resampling, then recheck validation performance without touching test.
